# Performance Risk Report

Generate full-cohort pass/fail risk predictions and highlight the students with the highest fail risk and their primary risk factors.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

base_dir = Path('../..').resolve()
sys.path.append(str(base_dir))

from src.predict import load_combined_performance_data, load_model, predict_students

processed_dir = base_dir / 'data' / 'processed' / 'performance'
model_path = base_dir / 'models' / 'performance' / 'pass_classifier_rf.joblib'


In [ ]:
data = load_combined_performance_data(base_dir)
feature_df = data.drop(columns=['G3', 'pass'])
model = load_model(model_path)
report_df = predict_students(feature_df, model=model)
report_df.insert(0, 'student_id', range(1, len(report_df) + 1))
report_df['actual_pass'] = data['pass'].reset_index(drop=True)
report_df['G3'] = data['G3'].reset_index(drop=True)
report_df['predicted_outcome'] = report_df['prediction'].map({0: 'Fail', 1: 'Pass'})
report_df = report_df.sort_values(by='risk_score', ascending=False).reset_index(drop=True)
report_df.head()


In [ ]:
output_path = processed_dir / 'student_predictions.csv'
report_df.to_csv(output_path, index=False)
print(f'Saved full performance risk report to: {output_path}')


In [ ]:
risk_columns = [
    'student_id',
    'predicted_outcome',
    'risk_score',
    'risk_level',
    'Primary_Risk_Factors',
    'G1',
    'G2',
    'G3',
    'absences',
    'studytime',
]
report_df.loc[report_df['risk_level'].isin(['High', 'Medium']), risk_columns].head(25)
